# From Course Homework to Spectral Hodge BME

**A progressive demo of pyBME's five model families**

This notebook walks from a classical spatial BME homework problem
(the kind assigned in ENVR 468/765 at UNC) through four increasingly
powerful extensions — each adding a new structural assumption about
the data-generating process.

| Part | Family | Key idea |
|------|--------|----------|
| 1 | **Spatial** | Fit a covariance model, make a BME map |
| 2 | **Space–time** | Separable $C = \sigma^2 C_s(r) C_t(\tau)$ |
| 3 | **Graph Laplacian** | The network *is* the prior — no covariance fitting |
| 4 | **Physics-informed** | Mass-balance constraints in the precision |
| 5 | **Spectral Hodge** | Time-varying operators, persistent spectral basis |

Parts 1–2 mirror the PM2.5 homework from Marc Serre's geostatistics
courses.  Parts 3–5 extend BME to domains where Euclidean distance
is the wrong metric and the graph structure itself carries information.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
%matplotlib inline

from pybme import (
    # Spatial / S-T
    bme_predict, bme_predict_st, fit_covariance,
    exponential_cov, eval_cov,
    SoftPDF, BMEResult,
    # Network
    adjacency_from_edges,
    NetworkCovariance, NetworkCovarianceST,
    bme_predict_network, bme_predict_network_st,
    # Physics-informed
    PhysicsInformedNetworkCovariance,
    build_graph_laplacian,
    # Hodge / Spectral
    build_oriented_incidence,
    HodgeNetworkCovariance, HodgeNetworkCovarianceST,
    SpectralHodgeNetworkCovariance, SpectralHodgeNetworkCovarianceST,
)

np.set_printoptions(precision=3, suppress=True)
print("pyBME imported successfully.")


---
## Part 1 — Classical Spatial BME  *(PM2.5 over California)*

> *"Given scattered monitoring stations with annual-average PM2.5,
> fit a covariance model and produce a BME map."*
> — ENVR 468 HW6 / ENVR 765 HW3

### Data

The California PM2.5 dataset (1997–2016) ships with pyBME.
115 EPA monitoring stations report annual-average PM$_{2.5}$
($\mu$g/m$^3$) with varying temporal coverage.  We pick **2010**
as the target year — a year with good station coverage.


In [ ]:
import os

data_file = os.path.join(os.path.dirname(os.getcwd()), "pybme", "examples", "pm2p5_CA_1997-2016.txt")
if not os.path.exists(data_file):
    # Fallback: try relative from notebook location
    data_file = os.path.join("..", "examples", "pm2p5_CA_1997-2016.txt")

# ── Load GeoEAS format ──────────────────────────────────────
with open(data_file) as f:
    _header = f.readline()
    n_cols = int(f.readline().strip())
    col_names = [f.readline().strip().rstrip(",") for _ in range(n_cols)]
    raw = np.loadtxt(f)

lon = raw[:, 3]
lat = raw[:, 4]
pm25_all = raw[:, 5:]  # columns 5–24 → years 1997–2016
pm25_all[pm25_all < -999] = np.nan  # missing-value sentinel

YEAR_IDX = 13  # 2010 (index 0 = 1997)
z_2010 = pm25_all[:, YEAR_IDX]
valid = ~np.isnan(z_2010)

print(f"Stations with data in 2010: {valid.sum()} / {len(valid)}")
print(f"PM2.5 range: {z_2010[valid].min():.1f} – {z_2010[valid].max():.1f} μg/m³")


### 1a. Exploratory scatter plot

Let's see where the stations are and what the PM$_{2.5}$ field looks like.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(lon[valid], lat[valid], c=z_2010[valid], cmap="YlOrRd",
                s=40, edgecolor="k", linewidth=0.3, vmin=4, vmax=20)
ax.scatter(lon[~valid], lat[~valid], c="grey", s=10, alpha=0.3, label="no data 2010")
plt.colorbar(sc, ax=ax, label="PM$_{2.5}$ (μg/m³)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("California PM$_{2.5}$ stations — 2010 annual average")
ax.legend(loc="lower left", fontsize=8)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### 1b. Fit a spatial covariance model

`fit_covariance` estimates sill, range, and nugget via restricted
maximum likelihood (REML).  We use an exponential model — the
network analogue of Matérn with $\nu = 1$.


In [ ]:
# Project lon/lat → km (local UTM-like projection around California centroid)
lon0, lat0 = lon[valid].mean(), lat[valid].mean()
x_km = (lon[valid] - lon0) * 111.32 * np.cos(np.radians(lat0))
y_km = (lat[valid] - lat0) * 110.57

ch = np.column_stack([x_km, y_km])
zh = z_2010[valid].copy()

fit = fit_covariance(ch, zh, model="exponential", order=0)
SILL = fit["sill"]
RANGE_S = fit["range"]
NUGGET = fit["nugget"]

print(f"Exponential fit:")
print(f"  sill    = {SILL:.2f} (μg/m³)²")
print(f"  range   = {RANGE_S:.1f} km")
print(f"  nugget  = {NUGGET:.2f} (μg/m³)²")
print(f"  NLL     = {fit['nll']:.2f}")


### 1c. BME prediction on a grid

With only **hard data** (exact measurements), BME reduces to
ordinary kriging.  We predict on a regular grid over California.


In [ ]:
# Build estimation grid
nx, ny = 40, 50
xg = np.linspace(x_km.min() - 20, x_km.max() + 20, nx)
yg = np.linspace(y_km.min() - 20, y_km.max() + 20, ny)
xx, yy = np.meshgrid(xg, yg)
ck = np.column_stack([xx.ravel(), yy.ravel()])

# Distance mask: only predict within 2× range of nearest station
from scipy.spatial import cKDTree
tree = cKDTree(ch)
dists, _ = tree.query(ck)
mask = dists < 2.0 * RANGE_S

ck_pred = ck[mask]
print(f"Predicting at {len(ck_pred)} grid points (of {len(ck)} total)")


In [ ]:
results = bme_predict(
    ck=ck_pred, ch=ch, zh=zh,
    model="exponential",
    params=[SILL, RANGE_S, NUGGET],
    nhmax=15,
    dmax=2.5 * RANGE_S,
    order=0,
)

means = np.array([r.mean for r in results])
stds = np.array([np.sqrt(r.variance) for r in results])
print(f"Predicted {len(results)} points.  Mean PM2.5: {means.mean():.2f} μg/m³")


In [ ]:
# Reconstruct full grid for plotting
mean_grid = np.full(len(ck), np.nan)
std_grid = np.full(len(ck), np.nan)
mean_grid[mask] = means
std_grid[mask] = stds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean map
im0 = axes[0].pcolormesh(xg, yg, mean_grid.reshape(ny, nx),
                          cmap="YlOrRd", vmin=4, vmax=20, shading="auto")
axes[0].scatter(x_km, y_km, c=zh, cmap="YlOrRd", s=20,
                edgecolor="k", linewidth=0.3, vmin=4, vmax=20)
plt.colorbar(im0, ax=axes[0], label="PM$_{2.5}$ (μg/m³)")
axes[0].set_title("BME posterior mean (2010)")
axes[0].set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
axes[0].set_aspect("equal")

# Std map
im1 = axes[1].pcolormesh(xg, yg, std_grid.reshape(ny, nx),
                          cmap="Blues", vmin=0, shading="auto")
axes[1].scatter(x_km, y_km, c="red", s=10)
plt.colorbar(im1, ax=axes[1], label="Posterior std (μg/m³)")
axes[1].set_title("BME posterior uncertainty (2010)")
axes[1].set_xlabel("Easting (km)")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()


### Part 1 — What we did

1. **Loaded** the PM2.5 monitoring data (GeoEAS format)
2. **Fitted** an exponential covariance model via REML
3. **Predicted** on a grid using `bme_predict` (= kriging with hard data only)
4. **Mapped** the posterior mean and uncertainty

This is the classical workflow that any BMElib/pyBME user knows.
Next, we add a temporal dimension.


---
## Part 2 — Space–Time BME

The PM2.5 dataset has annual values from 1997–2016.  Instead of
analysing one year at a time, the **space–time** family models the
joint covariance:

$$C(\mathbf{r}, \tau) = \sigma^2 \, C_s(\|\mathbf{r}\|) \, C_t(|\tau|)$$

This **separable** structure lets us borrow strength across time —
a station with data in 2009 and 2011 informs the 2010 estimate even
if its 2010 value is missing.


In [ ]:
# ── Build space–time arrays ─────────────────────────────────
years = np.arange(1997, 2017)  # 20 years

# Collect all valid (station, year) pairs
ch_st, th_st, zh_st = [], [], []
for i in range(len(lon)):
    xi = (lon[i] - lon0) * 111.32 * np.cos(np.radians(lat0))
    yi = (lat[i] - lat0) * 110.57
    for j, yr in enumerate(years):
        val = pm25_all[i, j]
        if not np.isnan(val):
            ch_st.append([xi, yi])
            th_st.append(float(yr))
            zh_st.append(val)

ch_st = np.array(ch_st)
th_st = np.array(th_st)
zh_st = np.array(zh_st)

print(f"S/T hard data points: {len(zh_st)}")
print(f"Year range: {th_st.min():.0f} – {th_st.max():.0f}")


In [ ]:
# ── Predict at a single station across all years ────────────
# Pick a station with incomplete coverage
n_valid = np.sum(~np.isnan(pm25_all), axis=1)
target_idx = np.argmin(np.abs(n_valid - 12))  # ~60% coverage
target_lon, target_lat = lon[target_idx], lat[target_idx]
target_x = (target_lon - lon0) * 111.32 * np.cos(np.radians(lat0))
target_y = (target_lat - lat0) * 110.57

ck_st = np.tile([target_x, target_y], (len(years), 1))
tk_st = years.astype(float)

# Temporal covariance: exponential with range ~4 years
results_st = bme_predict_st(
    ck=ck_st, tk=tk_st,
    ch=ch_st, th=th_st, zh=zh_st,
    model_s="exponential", params_s=[1.0, RANGE_S],
    model_t="exponential", params_t=[1.0, 4.0],
    sigma2=SILL,
    nhmax=20, nsmax=0,
    dmax_s=2.5 * RANGE_S, dmax_t=8.0,
    order=0,
)

means_st = np.array([r.mean for r in results_st])
stds_st = np.array([np.sqrt(r.variance) for r in results_st])


In [ ]:
# ── Plot the temporal profile ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

# Observed values at this station
obs_vals = pm25_all[target_idx, :]
obs_mask = ~np.isnan(obs_vals)

ax.fill_between(years, means_st - 2*stds_st, means_st + 2*stds_st,
                alpha=0.2, color="steelblue", label="95% CI")
ax.plot(years, means_st, "b-", lw=1.5, label="BME posterior mean")
ax.scatter(years[obs_mask], obs_vals[obs_mask], c="red", s=40,
           zorder=5, label="Observed")

ax.set_xlabel("Year")
ax.set_ylabel("PM$_{2.5}$ (μg/m³)")
ax.set_title(f"S/T BME at station ({target_lon:.2f}°, {target_lat:.2f}°) — "
             f"{int(obs_mask.sum())} of 20 years observed")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Part 2 — What changed

We went from $n = 87$ (one year, hard data) to $n = 1{,}200+$ observations
spread across 20 years.  The **separable S/T covariance** lets years
with data inform years without — filling temporal gaps at every station.

Both Parts 1 and 2 require **fitting a covariance model** to the data.
The model encodes our assumption about spatial/temporal correlation
structure.  What if the domain itself could encode that structure?


---
## Part 3 — The Operator Turn: Graph Laplacian Network BME

> *"What if the data lives on a river network, not in Euclidean space?"*

In Parts 1–2, we specified a covariance model (exponential, range = 200 km)
and fitted its parameters.  On a **network** — rivers, sewers, pipes — 
Euclidean distance is the wrong metric, and the adjacency structure
itself carries information about correlation.

The **graph Laplacian** $L = D - W$ (where $D$ = degree matrix,
$W$ = adjacency) encodes which nodes are connected.  The precision
matrix

$$Q = \frac{1}{\sigma^2}(\kappa^2 I + L)$$

defines a Gaussian Markov random field (GMRF) directly — no covariance
fitting needed.  The covariance is $C = \sigma^2(\kappa^2 I + L)^{-1}$.

This is the network analogue of the SPDE approach to Matérn fields
(Lindgren et al., 2011), applied to graph domains.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Build a synthetic river network
# ═══════════════════════════════════════════════════════════════
#
#     0 (headwater)     1 (headwater)
#      \               /
#       2 (junction)
#       |
#       3 (mid-reach)
#       |
#       4 (junction 2) ← 5 (tributary)
#       |
#       6 (lower reach)
#       |
#       7 (outlet)

n_nodes = 8
edges = np.array([
    [0, 2], [1, 2],   # headwaters merge
    [2, 3],            # main stem
    [3, 4],            # main stem
    [5, 4],            # tributary joins
    [4, 6],            # lower reach
    [6, 7],            # outlet
])

# Node positions for plotting (not used by BME — structure only)
node_xy = np.array([
    [0.0, 4.0],  # 0: headwater NW
    [4.0, 4.0],  # 1: headwater NE
    [2.0, 3.0],  # 2: junction
    [2.0, 2.0],  # 3: mid-reach
    [2.5, 1.0],  # 4: junction 2
    [4.5, 2.0],  # 5: tributary
    [2.5, 0.0],  # 6: lower reach
    [2.5,-1.0],  # 7: outlet
])

print(f"Network: {n_nodes} nodes, {len(edges)} edges")


In [ ]:
# ── Visualise the network ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
for i, j in edges:
    ax.plot([node_xy[i, 0], node_xy[j, 0]],
            [node_xy[i, 1], node_xy[j, 1]], "b-", lw=2, alpha=0.6)
    # Arrow showing flow direction
    mx = 0.6 * node_xy[j, 0] + 0.4 * node_xy[i, 0]
    my = 0.6 * node_xy[j, 1] + 0.4 * node_xy[i, 1]
    dx = node_xy[j, 0] - node_xy[i, 0]
    dy = node_xy[j, 1] - node_xy[i, 1]
    ax.annotate("", xy=(mx + 0.05*dx, my + 0.05*dy),
                xytext=(mx - 0.05*dx, my - 0.05*dy),
                arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5))

labels = ["HW-0", "HW-1", "Jct", "Mid", "Jct-2", "Trib", "Lower", "Outlet"]
for i, (x, y) in enumerate(node_xy):
    ax.plot(x, y, "o", ms=14, color="white", markeredgecolor="black", zorder=5)
    ax.text(x, y, str(i), ha="center", va="center", fontsize=8, fontweight="bold", zorder=6)
    ax.text(x + 0.2, y + 0.2, labels[i], fontsize=7, color="grey")

ax.set_title("Synthetic river network (8 nodes, 7 edges)")
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-1.5, 4.5)
ax.set_aspect("equal")
ax.axis("off")
plt.tight_layout()
plt.show()


### 3a. Build the graph Laplacian covariance

No parameter fitting.  The **adjacency matrix** defines the
correlation structure.  We only choose $\kappa$ (decorrelation
rate) and $\sigma^2$ (overall variance).


In [ ]:
# Build adjacency → NetworkCovariance (from_adjacency computes Laplacian)
W = adjacency_from_edges(n_nodes, edges)
net_cov = NetworkCovariance(W, kappa=1.0, sigma2=4.0, from_adjacency=True)

print("Laplacian L (dense):")
print(net_cov.L.toarray())
print()

print("Covariance matrix C = σ²(κ²I + L)⁻¹:")
C = net_cov.C_dense
print(np.array2string(C, precision=2, suppress_small=True))


In [ ]:
# ── Visualise the covariance matrix ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Covariance heatmap
im = axes[0].imshow(C, cmap="RdBu_r", vmin=-C.max(), vmax=C.max())
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Network covariance $C = \sigma^2(\kappa^2 I + L)^{-1}$")
axes[0].set_xlabel("Node")
axes[0].set_ylabel("Node")
for i in range(n_nodes):
    for j in range(n_nodes):
        axes[0].text(j, i, f"{C[i,j]:.2f}", ha="center", va="center", fontsize=7)

# Covariance from node 0 (headwater) to all others
axes[1].bar(range(n_nodes), C[0, :], color="steelblue", edgecolor="k")
axes[1].set_xlabel("Node")
axes[1].set_ylabel("Cov(node 0, node j)")
axes[1].set_title("Covariance from headwater (node 0)")
axes[1].set_xticks(range(n_nodes))
axes[1].set_xticklabels(labels, rotation=45, fontsize=8)

plt.tight_layout()
plt.show()


### 3b. BME prediction on the network

Observe concentrations at the headwaters and tributary (nodes 0, 1, 5).
Predict at all other nodes (junctions, mid-reach, outlet).


In [ ]:
# Observed nodes and values (e.g. pollutant concentration, mg/L)
ch_nodes = np.array([0, 1, 5])    # metered headwaters + tributary
zh_net = np.array([8.5, 3.2, 5.0])

# Predict at unmetered nodes
ck_nodes = np.array([2, 3, 4, 6, 7])

results_net = bme_predict_network(
    ck_nodes=ck_nodes, ch_nodes=ch_nodes, zh=zh_net,
    net_cov=net_cov,
    nhmax=8,
    order=0,
)

print(f"{'Node':>6s}  {'Label':>8s}  {'Mean':>8s}  {'Std':>8s}")
print("-" * 38)
for node, res in zip(ck_nodes, results_net):
    print(f"{node:6d}  {labels[node]:>8s}  {res.mean:8.2f}  {np.sqrt(res.variance):8.2f}")


In [ ]:
# ── Plot predictions on the network ─────────────────────────
all_vals = np.full(n_nodes, np.nan)
all_stds = np.full(n_nodes, np.nan)
for node, z in zip(ch_nodes, zh_net):
    all_vals[node] = z
    all_stds[node] = 0.0
for node, res in zip(ck_nodes, results_net):
    all_vals[node] = res.mean
    all_stds[node] = np.sqrt(res.variance)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax in axes:
    for i, j in edges:
        ax.plot([node_xy[i, 0], node_xy[j, 0]],
                [node_xy[i, 1], node_xy[j, 1]], "b-", lw=2, alpha=0.3)

# Mean
sc0 = axes[0].scatter(node_xy[:, 0], node_xy[:, 1], c=all_vals,
                       cmap="YlOrRd", s=200, edgecolor="k", zorder=5, vmin=3, vmax=9)
for i in range(n_nodes):
    axes[0].text(node_xy[i, 0], node_xy[i, 1] - 0.35,
                 f"{all_vals[i]:.1f}", ha="center", fontsize=8)
plt.colorbar(sc0, ax=axes[0], label="Concentration (mg/L)")
axes[0].set_title("Network BME — posterior mean")
# Mark observed nodes
axes[0].scatter(node_xy[ch_nodes, 0], node_xy[ch_nodes, 1],
                s=200, facecolors="none", edgecolors="limegreen", linewidths=2, zorder=6)
axes[0].set_aspect("equal"); axes[0].axis("off")

# Std
sc1 = axes[1].scatter(node_xy[:, 0], node_xy[:, 1], c=all_stds,
                       cmap="Blues", s=200, edgecolor="k", zorder=5)
for i in range(n_nodes):
    axes[1].text(node_xy[i, 0], node_xy[i, 1] - 0.35,
                 f"{all_stds[i]:.2f}", ha="center", fontsize=8)
plt.colorbar(sc1, ax=axes[1], label="Posterior std (mg/L)")
axes[1].set_title("Network BME — posterior uncertainty")
axes[1].scatter(node_xy[ch_nodes, 0], node_xy[ch_nodes, 1],
                s=200, facecolors="none", edgecolors="limegreen", linewidths=2, zorder=6)
axes[1].set_aspect("equal"); axes[1].axis("off")

plt.suptitle("Green rings = observed stations", fontsize=9, color="grey")
plt.tight_layout()
plt.show()


### Part 3 — What changed

We **never fitted a covariance model**.  The graph Laplacian
$L = D - W$ encodes the correlation structure directly.  The
precision matrix $Q \propto \kappa^2 I + L$ says:

- Nodes share a direct edge → conditionally correlated
- Nodes separated by many hops → weakly correlated
- The outlet (node 7) is naturally uncertain — it's far from all sensors

This is the **operator turn**: the domain structure *is* the prior.


---
## Part 4 — Physics-Informed Network BME

The graph Laplacian treats the network as **undirected** — it doesn't
know that water flows downstream.  The **physics-informed** family
adds a **mass-balance penalty** to the precision:

$$Q = \frac{1}{\sigma^2}\big(\kappa^2 I + \alpha L + \lambda H^\top H\big)$$

where $H$ is the directed mass-balance operator:

$$H x = \begin{pmatrix} x_2 - x_0 - x_1 \\ x_3 - x_2 \\ x_4 - x_3 - x_5 \\ x_6 - x_4 \\ x_7 - x_6 \end{pmatrix}$$

Each row says: *"the value at a junction should equal the sum of its
upstream parents."*  Higher $\lambda$ means the prior more strongly
favours flow-conserving fields.


In [ ]:
# Directed edges for mass-balance operator
directed_edges = edges.copy()  # already directed downstream

pi_cov = PhysicsInformedNetworkCovariance(
    laplacian=W,
    directed_edges=directed_edges,
    kappa=1.0,
    sigma2=4.0,
    alpha=1.0,    # undirected smoothness weight
    lam=2.0,      # mass-balance penalty weight
    from_adjacency=True,
)

print("Physics-informed precision Q:")
print(np.array2string(pi_cov.Q.toarray(), precision=2, suppress_small=True))
print()
print("Physics-informed covariance C:")
C_pi = pi_cov.C_dense
print(np.array2string(C_pi, precision=2, suppress_small=True))


In [ ]:
# ── Predict with physics-informed covariance ─────────────────
results_pi = bme_predict_network(
    ck_nodes=ck_nodes, ch_nodes=ch_nodes, zh=zh_net,
    net_cov=pi_cov,
    nhmax=8,
    order=0,
)

print(f"{'Node':>6s}  {'Label':>8s}  {'Laplacian':>10s}  {'Physics':>10s}  {'Δ':>6s}")
print("-" * 50)
for i, node in enumerate(ck_nodes):
    m_lap = results_net[i].mean
    m_pi = results_pi[i].mean
    print(f"{node:6d}  {labels[node]:>8s}  {m_lap:10.2f}  {m_pi:10.2f}  {m_pi - m_lap:+6.2f}")


In [ ]:
# ── Side-by-side comparison ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, title, results_list, cov_label in [
    (axes[0], "Graph Laplacian (Part 3)", results_net, "Undirected"),
    (axes[1], "Physics-Informed (Part 4)", results_pi, "Mass-balance"),
]:
    vals = np.full(n_nodes, np.nan)
    for node, z in zip(ch_nodes, zh_net):
        vals[node] = z
    for node, res in zip(ck_nodes, results_list):
        vals[node] = res.mean

    for i, j in edges:
        ax.plot([node_xy[i, 0], node_xy[j, 0]],
                [node_xy[i, 1], node_xy[j, 1]], "b-", lw=2, alpha=0.3)
    sc = ax.scatter(node_xy[:, 0], node_xy[:, 1], c=vals,
                    cmap="YlOrRd", s=200, edgecolor="k", zorder=5, vmin=3, vmax=9)
    for i in range(n_nodes):
        ax.text(node_xy[i, 0], node_xy[i, 1] - 0.35,
                f"{vals[i]:.1f}", ha="center", fontsize=8)
    ax.scatter(node_xy[ch_nodes, 0], node_xy[ch_nodes, 1],
               s=200, facecolors="none", edgecolors="limegreen", linewidths=2, zorder=6)
    ax.set_title(f"{title}\n({cov_label})")
    ax.set_aspect("equal"); ax.axis("off")

plt.tight_layout()
plt.show()


### Part 4 — What changed

The physics-informed model **pulls junction estimates toward
mass-balance consistency**.  At node 2 (where headwaters merge),
the prediction is closer to the sum-weighted average of nodes 0
and 1.  At node 4 (where the tributary joins), the tributary
measurement shapes the estimate.

The precision matrix $Q$ encodes both:
- **Smoothness** ($\alpha L$) — neighbours are correlated
- **Conservation** ($\lambda H^\top H$) — flow balance is encouraged


---
## Part 5 — Spectral Hodge: Time-Varying Operators

Real networks change over time.  River flows vary by season;
pumps turn on and off; storm events reroute flow.  When edge
weights $W_e(t)$ are time-varying, the Hodge Laplacian
$L_0(t) = B_1 W_e(t) B_1^\top$ also varies.

### The problem with naïve time-varying operators

If we recompute eigenvectors at each time step, the spectral
identity of each node rotates — destroying the persistent
structure that links metered and unmetered nodes.

### The spectral graph filter solution

`SpectralHodgeNetworkCovariance` fixes a **reference eigenbasis**
$V$ from a reference operator, then adapts eigenvalues to the
current hydraulic state via Galerkin projection:

$$C(t) = \sigma^2 V \,\mathrm{diag}\!\Big(\frac{1}{\kappa^2 + \lambda_i^{\mathrm{eff}}(t)}\Big) V^\top$$

When $L_0(t) = L_0^{ref}$, this is identical to the static case.


In [ ]:
# ── Build the Hodge machinery ────────────────────────────────
B1 = build_oriented_incidence(n_nodes, edges)

# Time-varying edge weights: simulate dry vs wet season
def edge_weights_at(t):
    """Seasonal edge weights: higher in wet season (winter)."""
    seasonal = 0.5 + 0.5 * np.sin(2 * np.pi * (t - 0.25))  # peaks at t=0.5
    base = np.ones(len(edges))
    # Headwater edges (0,1) get stronger in wet season
    base[0] *= (0.3 + 0.7 * seasonal)  # edge 0→2
    base[1] *= (0.3 + 0.7 * seasonal)  # edge 1→2
    # Tributary edge gets weaker in wet season
    base[4] *= (0.8 - 0.3 * seasonal)  # edge 5→4
    return np.maximum(base, 0.1)

print("Edge weights (dry, t=0.0):", edge_weights_at(0.0).round(2))
print("Edge weights (wet, t=0.5):", edge_weights_at(0.5).round(2))


In [ ]:
# Build SpectralHodgeNetworkCovariance
spectral_cov = SpectralHodgeNetworkCovariance(
    B1=B1,
    directed_edges=edges,
    edge_weight_func=edge_weights_at,
    kappa=1.0,
    sigma2=4.0,
    alpha=1.0,
    lam=1.5,
    n_modes=None,  # keep all modes (small network)
)

# Show how covariance changes with time
t_dry, t_wet = 0.0, 0.5
C_dry = spectral_cov.covariance_block_at(t_dry, np.arange(n_nodes), np.arange(n_nodes))
C_wet = spectral_cov.covariance_block_at(t_wet, np.arange(n_nodes), np.arange(n_nodes))

print("Covariance at node 2 (junction):")
print(f"  Dry season: Cov(0,2)={C_dry[0,2]:.3f}, Cov(1,2)={C_dry[1,2]:.3f}, Cov(5,2)={C_dry[5,2]:.3f}")
print(f"  Wet season: Cov(0,2)={C_wet[0,2]:.3f}, Cov(1,2)={C_wet[1,2]:.3f}, Cov(5,2)={C_wet[5,2]:.3f}")


In [ ]:
# ── Visualise the spectral basis ─────────────────────────────
V = spectral_cov.V  # eigenvectors (N × n_modes)
n_show = min(4, V.shape[1])

fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for k, ax in enumerate(axes):
    for i, j in edges:
        ax.plot([node_xy[i, 0], node_xy[j, 0]],
                [node_xy[i, 1], node_xy[j, 1]], "grey", lw=1, alpha=0.5)
    vk = V[:, k]
    sc = ax.scatter(node_xy[:, 0], node_xy[:, 1], c=vk,
                    cmap="RdBu_r", s=150, edgecolor="k", zorder=5,
                    vmin=-np.abs(vk).max(), vmax=np.abs(vk).max())
    ax.set_title(f"Mode {k}\n(λ={spectral_cov.lam_ref[k]:.2f})")
    ax.set_aspect("equal"); ax.axis("off")

plt.suptitle("Reference spectral basis — persistent node identity", fontsize=11)
plt.tight_layout()
plt.show()


### 5a. Space–time network prediction

We place observations at three time points (dry, transition, wet)
and predict across all nodes and times.


In [ ]:
# ── Build S/T spectral Hodge covariance ──────────────────────
spectral_cov_st = SpectralHodgeNetworkCovarianceST(
    spectral_cov=spectral_cov,
    model_t="exponential",
    params_t=[1.0, 0.3],  # temporal range = 0.3 (seasonal scale)
    sigma2=4.0,
)

# Observations: 3 stations × 2 times
ch_st_net = np.array([0, 1, 5, 0, 1, 5])           # metered nodes
th_st_net = np.array([0.0, 0.0, 0.0, 0.5, 0.5, 0.5])  # dry + wet
zh_st_net = np.array([8.5, 3.2, 5.0, 12.0, 5.5, 3.5])  # dry + wet values

# Predict at all unmetered nodes at 3 times
pred_times = np.array([0.0, 0.25, 0.5])
ck_rep = np.tile(ck_nodes, len(pred_times))
tk_rep = np.repeat(pred_times, len(ck_nodes))

results_spectral_st = bme_predict_network_st(
    ck_nodes=ck_rep, tk=tk_rep,
    ch_nodes=ch_st_net, th=th_st_net, zh=zh_st_net,
    net_cov_st=spectral_cov_st,
    nhmax=8,
    order=0,
)

print(f"{'Time':>6s}  {'Node':>6s}  {'Label':>8s}  {'Mean':>8s}  {'Std':>8s}")
print("-" * 44)
for idx, (node, t) in enumerate(zip(ck_rep, tk_rep)):
    r = results_spectral_st[idx]
    print(f"{t:6.2f}  {node:6d}  {labels[node]:>8s}  {r.mean:8.2f}  {np.sqrt(r.variance):8.2f}")


In [ ]:
# ── Plot S/T predictions ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
time_labels = ["Dry (t=0.0)", "Transition (t=0.25)", "Wet (t=0.5)"]

for tidx, (t_val, ax, tlbl) in enumerate(zip(pred_times, axes, time_labels)):
    vals = np.full(n_nodes, np.nan)
    # Observed values at this time
    for k, (node, t_obs) in enumerate(zip(ch_st_net, th_st_net)):
        if abs(t_obs - t_val) < 0.01:
            vals[node] = zh_st_net[k]
    # Predicted values
    for k in range(len(ck_nodes)):
        res_idx = tidx * len(ck_nodes) + k
        vals[ck_nodes[k]] = results_spectral_st[res_idx].mean

    for i, j in edges:
        ax.plot([node_xy[i, 0], node_xy[j, 0]],
                [node_xy[i, 1], node_xy[j, 1]], "b-", lw=2, alpha=0.3)
    sc = ax.scatter(node_xy[:, 0], node_xy[:, 1], c=vals,
                    cmap="YlOrRd", s=200, edgecolor="k", zorder=5, vmin=3, vmax=13)
    for i in range(n_nodes):
        if not np.isnan(vals[i]):
            ax.text(node_xy[i, 0], node_xy[i, 1] - 0.35,
                    f"{vals[i]:.1f}", ha="center", fontsize=8)
    ax.scatter(node_xy[ch_nodes, 0], node_xy[ch_nodes, 1],
               s=200, facecolors="none", edgecolors="limegreen", linewidths=2, zorder=6)
    ax.set_title(tlbl)
    ax.set_aspect("equal"); ax.axis("off")

plt.suptitle("Spectral Hodge S/T BME — time-varying network covariance", fontsize=12)
plt.tight_layout()
plt.show()


---
## Part 6 — Cross-Method Comparison

Let's compare the three network methods (Parts 3–5) on the
same static estimation problem from Part 3 (nodes 0, 1, 5
observed; predict 2, 3, 4, 6, 7).


In [ ]:
# ── Static comparison: Laplacian vs Physics-Informed vs Spectral ──
# Spectral at t=0 (dry season, close to unit weights)
results_spectral = bme_predict_network(
    ck_nodes=ck_nodes, ch_nodes=ch_nodes, zh=zh_net,
    net_cov=spectral_cov,  # uses covariance_block (static fallback at t=0)
    nhmax=8,
    order=0,
)

# Build comparison table
methods = ["Graph Laplacian", "Physics-Informed", "Spectral Hodge"]
all_results = [results_net, results_pi, results_spectral]

print(f"{'Node':>6s}  {'Label':>8s}", end="")
for m in methods:
    print(f"  {m:>16s}", end="")
print()
print("-" * (18 + 18 * len(methods)))

for i, node in enumerate(ck_nodes):
    print(f"{node:6d}  {labels[node]:>8s}", end="")
    for res_list in all_results:
        r = res_list[i]
        print(f"    {r.mean:5.2f} ± {np.sqrt(r.variance):.2f}", end="")
    print()


In [ ]:
# ── Bar chart comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(ck_nodes))
width = 0.25
colors = ["#4C72B0", "#55A868", "#C44E52"]

for idx, (method, res_list, color) in enumerate(zip(methods, all_results, colors)):
    means_m = [res_list[i].mean for i in range(len(ck_nodes))]
    stds_m = [np.sqrt(res_list[i].variance) for i in range(len(ck_nodes))]
    axes[0].bar(x + idx * width, means_m, width, label=method, color=color,
                edgecolor="k", linewidth=0.5)
    axes[1].bar(x + idx * width, stds_m, width, label=method, color=color,
                edgecolor="k", linewidth=0.5)

for ax, ylabel, title in [
    (axes[0], "Posterior mean (mg/L)", "Predicted concentration"),
    (axes[1], "Posterior std (mg/L)", "Prediction uncertainty"),
]:
    ax.set_xticks(x + width)
    ax.set_xticklabels([labels[n] for n in ck_nodes], fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


---
## Summary

| Part | Family | Covariance source | Key assumption |
|------|--------|-------------------|----------------|
| 1 | Spatial | Fitted exponential $C(h)$ | Isotropy, stationarity |
| 2 | Space–time | Separable $C_s \cdot C_t$ | Temporal stationarity |
| 3 | Graph Laplacian | $(\kappa^2 I + L)^{-1}$ | Network topology only |
| 4 | Physics-informed | $(\kappa^2 I + \alpha L + \lambda H^\top H)^{-1}$ | Mass-balance / flow conservation |
| 5 | Spectral Hodge | $V \,\mathrm{diag}(1/(\kappa^2 + \lambda_i^{eff}(t)))\, V^\top$ | Persistent spectral basis, time-varying hydraulics |

### The progression

1. **Parts 1–2**: Classical geostatistics — you must *specify and fit* a covariance model.
2. **Part 3**: The operator turn — the graph Laplacian *is* the prior.  No fitting needed.
3. **Part 4**: Physics enriches topology — directed flow constraints shape the posterior.
4. **Part 5**: Time-varying operators preserve spectral node identity while adapting to changing hydraulics.

Each step adds structural information to the prior, reducing dependence
on parametric covariance assumptions and increasing the physical
interpretability of the posterior.

---

*Built with [pyBME](https://github.com/your-repo/pybme) v0.5.0*
